In [2]:
import pandas as pd

# Load the cleaned datasets
df_crashes = pd.read_csv('cleaned_crashes.csv')
df_persons = pd.read_csv('cleaned_persons.csv')



In [3]:
# Merge on COLLISION_ID
df_merged = pd.merge(df_crashes, df_persons, on='COLLISION_ID', how='left')

# Quick check
print("First 5 rows of merged data:")
print(df_merged.head())

print("\nShape of merged data:", df_merged.shape)

First 5 rows of merged data:
   CRASH DATE CRASH TIME   BOROUGH   LATITUDE  LONGITUDE  \
0  09/11/2021       2:39  BROOKLYN  40.693314 -73.933238   
1  09/11/2021       2:39  BROOKLYN  40.693314 -73.933238   
2  09/11/2021       2:39  BROOKLYN  40.693314 -73.933238   
3  09/11/2021       2:39  BROOKLYN  40.693314 -73.933238   
4  03/26/2022      11:45  BROOKLYN  40.592410 -73.788995   

                          LOCATION           ON STREET NAME  \
0  (40.693314, -73.93323799999999)    WHITESTONE EXPRESSWAY   
1  (40.693314, -73.93323799999999)    WHITESTONE EXPRESSWAY   
2  (40.693314, -73.93323799999999)    WHITESTONE EXPRESSWAY   
3  (40.693314, -73.93323799999999)    WHITESTONE EXPRESSWAY   
4           (40.59241, -73.788995)  QUEENSBORO BRIDGE UPPER   

                  CROSS STREET NAME                           OFF STREET NAME  \
0                         20 AVENUE  PARKING LOT 110-00 ROCKAWAY BOULEVARD      
1                         20 AVENUE  PARKING LOT 110-00 ROCKAWAY BOUL

In [4]:

import os
import pandas as pd

# If df_merged already exists in the notebook, we'll use it.
# Otherwise try to find an integrated CSV in the same folder as your uploaded notebook.
if 'df_merged' not in globals():
    # path of the uploaded notebook (from your session history)
    notebook_path = "/mnt/data/7654fd42-158a-489c-a0a6-78fbe4268de3.ipynb"
    notebook_dir = os.path.dirname(notebook_path)
    # look for likely integrated CSV filenames in the notebook folder
    candidates = [
        os.path.join(notebook_dir, "integrated.csv"),
        os.path.join(notebook_dir, "df_merged.csv"),
        os.path.join(notebook_dir, "merged.csv"),
        os.path.join(notebook_dir, "Data_Engneering_Project_1.csv"),
    ]
    found = None
    for c in candidates:
        if os.path.exists(c):
            found = c
            break

    if found:
        print(f"Loading merged dataframe from: {found}")
        df_merged = pd.read_csv(found)
    else:
        raise FileNotFoundError(
            "df_merged is not defined and no integrated CSV found in the notebook directory. "
            "Either define df_merged earlier or save your merged data as one of: "
            + ", ".join(candidates)
        )

# Copy data frame
df_clean = df_merged.copy()

# 1. HANDLE NEW MISSING VALUES INTRODUCED BY THE JOIN

# persons columns might be missing if no people were recorded in the crash
person_columns = [col for col in df_clean.columns if col.upper().startswith("PERSON") or col.upper().startswith("PED")]

# fill numeric person-related columns with 0 (no recorded injury/fatality)
num_person_cols = df_clean[person_columns].select_dtypes(include=["float64", "int64", "int32", "float32"]).columns
df_clean[num_person_cols] = df_clean[num_person_cols].fillna(0)

# fill categorical person-related columns with "Unknown" or "Not Specified"
cat_person_cols = df_clean[person_columns].select_dtypes(include=["object", "string"]).columns
df_clean[cat_person_cols] = df_clean[cat_person_cols].fillna("Unknown")

# any crash columns that are missing after the join are a red flag → fill with sensible defaults
if "BOROUGH" in df_clean.columns:
    df_clean["BOROUGH"] = df_clean["BOROUGH"].fillna("Unknown")
if "LOCATION" in df_clean.columns:
    df_clean["LOCATION"] = df_clean["LOCATION"].fillna("(0.0, 0.0)")
if "ZIP_CODE" in df_clean.columns:
    df_clean["ZIP_CODE"] = df_clean["ZIP_CODE"].fillna("00000")

# 2. REMOVE INCONSISTENT, DUPLICATE, OR REDUNDANT COLUMNS

# If join created duplicated columns like LATITUDE_x / LATITUDE_y
redundant = [col for col in df_clean.columns if col.endswith("_x") or col.endswith("_y")]

# Keep one version → usually “_x” comes from df_crashes
for col in redundant:
    base = col.replace("_x", "").replace("_y", "")
    # if a base column already exists, we don't rename (prefer existing)
    if base in df_clean.columns:
        continue
    df_clean.rename(columns={col: base}, inplace=True)

# If both _x and _y exist: prefer the crash table's version (_x)
to_drop = [col for col in df_clean.columns if col.endswith("_y")]
df_clean.drop(columns=to_drop, inplace=True, errors="ignore")

# Remove exact duplicates if any occurred after merging
df_clean.drop_duplicates(inplace=True)

# 3. FIX DATATYPE MISMATCHES

# Convert crash date + time fields to datetime
if "CRASH_DATE" in df_clean.columns:
    df_clean["CRASH_DATE"] = pd.to_datetime(df_clean["CRASH_DATE"], errors="coerce")

if "CRASH_TIME" in df_clean.columns:
    # If CRASH_TIME is like "13:45" this will coerce to today's date + time; keep only time if you want:
    df_clean["CRASH_TIME"] = pd.to_datetime(df_clean["CRASH_TIME"], format="%H:%M", errors="coerce").dt.time

# Convert coordinates to numeric
for col in ["LATITUDE", "LONGITUDE"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Convert injury/fatality counts to integers
injury_cols = [c for c in df_clean.columns if "INJUR" in c.upper() or "FATAL" in c.upper()]
for col in injury_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0).astype(int)

# Create YEAR column if it doesn’t exist
if "YEAR" not in df_clean.columns and "CRASH_DATE" in df_clean.columns:
    df_clean["YEAR"] = df_clean["CRASH_DATE"].dt.year

# Final check for remaining missing values (for debugging)
missing_summary = df_clean.isna().sum().sort_values(ascending=False).head(20)
print("Top missing-value counts after cleaning:")
print(missing_summary)

# --- EXPORT cleaned integrated CSV (safe, creates directory if needed) ---
import os

# Option A: prefer the notebook folder if we detected it earlier, else use current working dir
# If you used the previous cell, `notebook_path` may already be defined. If not, we fall back.
notebook_path = globals().get("notebook_path", r"/mnt/data/7654fd42-158a-489c-a0a6-78fbe4268de3.ipynb")
notebook_dir = os.path.dirname(notebook_path) if notebook_path else ""

# If that dir doesn't exist on this system (Windows, etc.), use current working directory
if not notebook_dir or not os.path.isdir(notebook_dir):
    output_dir = os.getcwd()
else:
    output_dir = notebook_dir

# Ensure directory exists
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "df_clean_integrated.csv")

# Write CSV
df_clean.to_csv(output_path, index=False)
print(f"Cleaned integrated CSV written to: {output_path}")
print("Output directory contents (first 20 items):")
print(os.listdir(output_dir)[:20])


Top missing-value counts after cleaning:
YEAR                   289761
SAFETY_EQUIPMENT       289761
CRASH_DATE             289761
CRASH_TIME             289761
VEHICLE_ID             289761
EJECTION               289761
POSITION_IN_VEHICLE    289761
EMOTIONAL_STATUS       289761
COMPLAINT              289761
UNIQUE_ID              289761
LONGITUDE               22486
LATITUDE                22486
BODILY_INJURY               0
LOC_CLUSTER                 0
PED_ROLE                    0
PERSON_AGE                  0
PERSON_INJURY               0
PERSON_TYPE                 0
PERSON_ID                   0
PERSON_SEX                  0
dtype: int64
Cleaned integrated CSV written to: /Users/refkarezkall/visualization ber/df_clean_integrated.csv
Output directory contents (first 20 items):
['merging_CleaningAfterMerge_datasets.ipynb', 'Data_Engneering_Project_1.ipynb', 'crashes_dataset_cleaning.ipynb', 'df_clean_integrated.csv', 'cleaned_persons.csv', 'cleaned_crashes.csv', 'persons_dataset_

In [5]:
import pandas as pd

# Adjust your username if needed
df = pd.read_csv('df_clean_integrated.csv', low_memory=False)

df.head()

,CRASH DATE,CRASH TIME,BOROUGH,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,...,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,COMPLAINT,PED_ROLE,PERSON_SEX,YEAR
0,09/11/2021,2:39,BROOKLYN,40.693314,-73.933238,"(40.693314, -73.93323799999999)",WHITESTONE EXPRESSWAY,20 AVENUE,PARKING LOT 110-00 ROCKAWAY BOULEVARD,2,...,29.0,not ejected,conscious,0,driver,lap belt,complaint of pain or nausea,driver,m,2021.0
1,09/11/2021,2:39,BROOKLYN,40.693314,-73.933238,"(40.693314, -73.93323799999999)",WHITESTONE EXPRESSWAY,20 AVENUE,PARKING LOT 110-00 ROCKAWAY BOULEVARD,2,...,36.0,missing,missing,0,missing,missing,missing,registrant,missing,2021.0
2,09/11/2021,2:39,BROOKLYN,40.693314,-73.933238,"(40.693314, -73.93323799999999)",WHITESTONE EXPRESSWAY,20 AVENUE,PARKING LOT 110-00 ROCKAWAY BOULEVARD,2,...,25.0,missing,missing,0,missing,missing,missing,registrant,m,2021.0
3,09/11/2021,2:39,BROOKLYN,40.693314,-73.933238,"(40.693314, -73.93323799999999)",WHITESTONE EXPRESSWAY,20 AVENUE,PARKING LOT 110-00 ROCKAWAY BOULEVARD,2,...,33.0,not ejected,conscious,0,"front passenger, if two or more persons, inclu...",lap belt,complaint of pain or nausea,passenger,m,2021.0
4,03/26/2022,11:45,BROOKLYN,40.592410,-73.788995,"(40.59241, -73.788995)",QUEENSBORO BRIDGE UPPER,HORACE HARDING EXPRESSWAY,PARKING LOT 110-00 ROCKAWAY BOULEVARD,1,...,28.0,not ejected,conscious,0,driver,lap belt & harness,complaint of pain or nausea,driver,f,2022.0


In [6]:
df.columns.tolist()


['CRASH DATE',
 'CRASH TIME',
 'BOROUGH',
 'LATITUDE',
 'LONGITUDE',
 'LOCATION',
 'ON STREET NAME',
 'CROSS STREET NAME',
 'OFF STREET NAME',
 'NUMBER OF PERSONS INJURED',
 'NUMBER OF PERSONS KILLED',
 'NUMBER OF PEDESTRIANS INJURED',
 'NUMBER OF PEDESTRIANS KILLED',
 'NUMBER OF CYCLIST INJURED',
 'NUMBER OF CYCLIST KILLED',
 'NUMBER OF MOTORIST INJURED',
 'NUMBER OF MOTORIST KILLED',
 'CONTRIBUTING FACTOR VEHICLE 1',
 'COLLISION_ID',
 'VEHICLE TYPE CODE 1',
 'CRASH_DATETIME',
 'LOC_CLUSTER',
 'UNIQUE_ID',
 'CRASH_DATE',
 'CRASH_TIME',
 'PERSON_ID',
 'PERSON_TYPE',
 'PERSON_INJURY',
 'VEHICLE_ID',
 'PERSON_AGE',
 'EJECTION',
 'EMOTIONAL_STATUS',
 'BODILY_INJURY',
 'POSITION_IN_VEHICLE',
 'SAFETY_EQUIPMENT',
 'COMPLAINT',
 'PED_ROLE',
 'PERSON_SEX',
 'YEAR']

In [7]:
df['CRASH_DATETIME'] = pd.to_datetime(df['CRASH_DATETIME'], errors='coerce')


In [8]:
df["year"] = df['CRASH_DATETIME'].dt.year
df["month"] = df['CRASH_DATETIME'].dt.month
df["day"] = df['CRASH_DATETIME'].dt.day_name()
df["hour"] = df['CRASH_DATETIME'].dt.hour
